# Carte du réseau piéton

Lit les tables `silver.reseau_pieton_troncons` et `silver.reseau_pieton_points` sur une zone, puis affiche :
- le **réseau marchable**, coloré par famille de voies ;
- en **orange**, ce qu'on atteint **réellement en 5 minutes à pied** en suivant les chemins ;
- en **pointillés rouges**, le cercle de 400 m « à vol d'oiseau », pour comparer.

L'écart entre les deux montre pourquoi on a besoin du réseau piéton : rivières, voies ferrées ou îlots fermés obligent à des détours.

⚠️ `lh_mobility_citiz` doit être le lakehouse **par défaut**, et la région du point choisi doit avoir été traitée par le notebook silver.

In [1]:
%pip install folium networkx --quiet

## 1. Paramètres
Change `LIEU`, `CENTRE_LAT` et `CENTRE_LON` pour tester un autre endroit (coordonnées visibles en cliquant droit sur Google Maps).

In [2]:
SCHEMA = "silver"

# Point de départ de la démonstration (par défaut : place Kléber, Strasbourg)
LIEU       = "Place Kléber, Strasbourg"
CENTRE_LAT = 48.5834
CENTRE_LON = 7.7456

RAYON_CARTE_M  = 1000   # taille de la zone affichée autour du point
MINUTES_MARCHE = 5
VITESSE_KMH    = 4.8    # vitesse de marche moyenne
DISTANCE_MAX_M = VITESSE_KMH * 1000 / 60 * MINUTES_MARCHE   # 5 min à 4,8 km/h = 400 m

DOSSIER_CARTES = "/lakehouse/default/Files/silver/osm/cartes"

## 2. Lecture des tables silver sur la zone

In [3]:
import math
from pyspark.sql import functions as F

# Rectangle autour du point (1 degré de latitude ≈ 111 km)
dlat = RAYON_CARTE_M / 111_320
dlon = RAYON_CARTE_M / (111_320 * math.cos(math.radians(CENTRE_LAT)))

def dans_zone(col_lat, col_lon):
    return (F.col(col_lat).between(CENTRE_LAT - dlat, CENTRE_LAT + dlat)
            & F.col(col_lon).between(CENTRE_LON - dlon, CENTRE_LON + dlon))

troncons = (spark.table(f"{SCHEMA}.reseau_pieton_troncons")
            .filter(dans_zone("depart_latitude", "depart_longitude")
                    | dans_zone("arrivee_latitude", "arrivee_longitude"))
            .select("id_point_depart", "id_point_arrivee", "depart_longitude", "depart_latitude",
                    "arrivee_longitude", "arrivee_latitude", "longueur_m", "type_voie", "nom_voie")
            .toPandas())

points = (spark.table(f"{SCHEMA}.reseau_pieton_points")
          .filter(dans_zone("latitude", "longitude"))
          .toPandas())

print(f"{len(troncons):,} tronçons et {len(points):,} points dans la zone")
if len(troncons) > 40_000:
    print("⚠️ Zone très dense : la carte risque d'être lente. Réduis RAYON_CARTE_M (ex. 600).")
if troncons.empty:
    raise ValueError("Aucune donnée dans cette zone : la région a-t-elle bien été traitée par NB_12 ?")

## 3. Calcul de la zone atteignable en 5 minutes

In [4]:
import networkx as nx
import numpy as np

def distance_m(lat1, lon1, lat2, lon2):
    """Distance à vol d'oiseau en mètres (fonctionne sur des colonnes entières)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6_371_008.8 * np.arcsin(np.sqrt(a))

# 1) Le graphe : chaque tronçon relie deux points, avec sa longueur
G = nx.Graph()
G.add_weighted_edges_from(zip(troncons.id_point_depart, troncons.id_point_arrivee, troncons.longueur_m),
                          weight="longueur_m")

# 2) Le point du réseau le plus proche du lieu choisi
points["dist_vol_oiseau_m"] = distance_m(CENTRE_LAT, CENTRE_LON, points.latitude, points.longitude)
depart = points.loc[points.dist_vol_oiseau_m.idxmin()]
print(f"Départ : point {depart.id_point}, à {depart.dist_vol_oiseau_m:.0f} m du lieu choisi")

# 3) On « marche » le long des chemins jusqu'à DISTANCE_MAX_M (algorithme de Dijkstra, celui des GPS)
atteints = nx.single_source_dijkstra_path_length(G, depart.id_point, cutoff=DISTANCE_MAX_M, weight="longueur_m")
troncons["atteint"] = troncons.id_point_depart.isin(atteints) & troncons.id_point_arrivee.isin(atteints)

# 4) Comparaison avec le cercle « à vol d'oiseau »
dans_cercle = points[points.dist_vol_oiseau_m <= DISTANCE_MAX_M]
part = dans_cercle.id_point.isin(atteints).mean() * 100
km_atteints = troncons.loc[troncons.atteint, "longueur_m"].sum() / 1000

print(f"Réseau parcourable en {MINUTES_MARCHE} min : {km_atteints:.1f} km de chemins")
print(f"Points à moins de {DISTANCE_MAX_M:.0f} m à vol d'oiseau : {len(dans_cercle):,}")
print(f"… dont réellement atteignables en {MINUTES_MARCHE} min à pied : {part:.0f} %")

## 4. Carte

In [5]:
import os, unicodedata, folium

FAMILLES = {   # regroupement des types de voie pour une légende lisible
    "Cheminements piétons": ("#1a9850", ["Chemin piéton", "Rue piétonne", "Escalier", "Zone de rencontre", "Sentier"]),
    "Rues":                 ("#4575b4", ["Rue résidentielle", "Voie de desserte", "Petite route", "Voie non qualifiée"]),
    "Routes":               ("#555555", ["Route locale", "Route secondaire", "Route principale", "Voie rapide ouverte aux piétons"]),
    "Chemins ruraux et pistes cyclables": ("#a6611a", ["Chemin agricole ou forestier", "Chemin équestre",
                                                        "Piste cyclable ouverte aux piétons"]),
}
def fr(x, dec=0):
    """Nombre au format français : 1 942 ; 9,5"""
    return f"{x:,.{dec}f}".replace(",", " ").replace(".", ",")

famille_de = {t: f for f, (_, types) in FAMILLES.items() for t in types}
troncons["famille"] = troncons.type_voie.map(famille_de).fillna("Autres")
COULEURS = {f: c for f, (c, _) in FAMILLES.items()} | {"Autres": "#999999"}

def geojson(df):
    return {"type": "FeatureCollection", "features": [
        {"type": "Feature",
         "geometry": {"type": "LineString", "coordinates": [[round(r.depart_longitude, 6), round(r.depart_latitude, 6)],
                                                            [round(r.arrivee_longitude, 6), round(r.arrivee_latitude, 6)]]},
         "properties": {"type": r.type_voie, "nom": r.nom_voie or "—", "longueur": f"{r.longueur_m:.0f} m"}}
        for r in df.itertuples()]}

infobulle = lambda: folium.GeoJsonTooltip(fields=["type", "nom", "longueur"],
                                          aliases=["Type de voie", "Nom", "Longueur"])

m = folium.Map(location=[CENTRE_LAT, CENTRE_LON], zoom_start=16, tiles=None)
folium.TileLayer("OpenStreetMap", opacity=0.45, control=False).add_to(m)   # fond de carte éclairci

# Calque 1 : le réseau piéton, une couleur par famille de voies
for famille, df in troncons.groupby("famille"):
    folium.GeoJson(geojson(df), name=f"{famille} ({fr(len(df))} tronçons)", tooltip=infobulle(),
                   style_function=lambda _, c=COULEURS[famille]: {"color": c, "weight": 2, "opacity": 0.7}
                   ).add_to(m)

# Calque 2 : ce qui est atteignable en 5 minutes en suivant les chemins
folium.GeoJson(geojson(troncons[troncons.atteint]), name=f"Atteignable en {MINUTES_MARCHE} min à pied",
               tooltip=infobulle(),
               style_function=lambda _: {"color": "#f46d43", "weight": 6, "opacity": 0.9}).add_to(m)

# Calque 3 : le cercle « naïf » à vol d'oiseau
folium.Circle([CENTRE_LAT, CENTRE_LON], radius=DISTANCE_MAX_M, color="#d73027", weight=2,
              dash_array="8 6", fill=False, name=f"Cercle de {DISTANCE_MAX_M:.0f} m à vol d'oiseau").add_to(m)

# Calque 4 (masqué par défaut) : impasses et carrefours
for type_point, couleur in (("Carrefour", "#2c7bb6"), ("Impasse", "#d7191c")):
    sel = points[points.type_point == type_point]
    fc = {"type": "FeatureCollection", "features": [
        {"type": "Feature", "properties": {"type": type_point},
         "geometry": {"type": "Point", "coordinates": [round(r.longitude, 6), round(r.latitude, 6)]}}
        for r in sel.itertuples()]}
    folium.GeoJson(fc, name=f"{type_point}s ({fr(len(sel))})", show=False,
                   marker=folium.CircleMarker(radius=3, weight=0, fill=True, fill_opacity=0.9),
                   style_function=lambda _, c=couleur: {"fillColor": c, "color": c},
                   tooltip=folium.GeoJsonTooltip(fields=["type"], labels=False)).add_to(m)

folium.Marker([depart.latitude, depart.longitude], tooltip=f"Départ : {LIEU}",
              icon=folium.Icon(color="orange", icon="user", prefix="fa")).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

# Encadré explicatif
lignes = "".join(f'<div><span style="display:inline-block;width:18px;height:4px;background:{COULEURS[f]};'
                 f'margin-right:6px;vertical-align:middle"></span>{f}</div>' for f in COULEURS)
encadre = f"""
<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 14px;
            border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);font:13px/1.5 sans-serif;max-width:300px">
  <b>Réseau piéton autour de : {LIEU}</b><br>
  <span style="color:#f46d43"><b>━━</b></span> atteignable en {MINUTES_MARCHE} min à pied ({fr(km_atteints, 1)} km de chemins)<br>
  <span style="color:#d73027"><b>┅┅</b></span> cercle de {DISTANCE_MAX_M:.0f} m à vol d'oiseau<br>
  <b>{fr(part)} %</b> des points du cercle sont réellement atteignables en {MINUTES_MARCHE} min
  <hr style="margin:6px 0">{lignes}
  <div style="color:#777;margin-top:6px">Données © contributeurs OpenStreetMap (ODbL)</div>
</div>"""
m.get_root().html.add_child(folium.Element(encadre))

# Sauvegarde dans le lakehouse (fichier HTML ouvrable dans n'importe quel navigateur)
os.makedirs(DOSSIER_CARTES, exist_ok=True)
lieu_court = unicodedata.normalize("NFKD", LIEU.split(",")[0]).encode("ascii", "ignore").decode().lower()
nom_fichier = f"{DOSSIER_CARTES}/carte_osm_reseau_pieton_{lieu_court.replace(' ', '_')}.html"
m.save(nom_fichier)
print("Carte enregistrée :", nom_fichier.replace("/lakehouse/default/", ""))

m